# V-JEPA 2.1 × OLMo-Earth Fine-Tuning Notebook

Self-supervised JEPA fine-tuning on OLMo-Earth Sentinel-2 monthly composites.

**Sections:**
1. Paths & configuration
2. Build model + load checkpoint
3. Dataset + DataLoader
4. Sanity check (1 forward pass — expect loss ≈ 0.29)
5. Training (3 stages with tqdm + loss history)
6. Loss curve
7. CLI command for production runs
8. Post-training: load fine-tuned checkpoint + PCA embeddings

In [ ]:
from pathlib import Path

# ── edit these two paths before running ───────────────────────────────────
CKPT_PATH  = "./vjepa2_1_vitl_dist_vitG_384.pt"       # pretrained V-JEPA 2.1 ViT-L
TAR_GLOB   = "*.tar"                                   # OLMo-Earth 10_sentinel2_l2a_monthly
# ─────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "./checkpoints_olmoearth"
CONFIG_YML = "vjepa2/configs/finetune/vitl16/olmoearth-256px-12f.yaml"

# None → full epoch (~17 830 steps); int → step cap per epoch for smoke test
STEPS_CAP = 50

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Output → {Path(OUTPUT_DIR).resolve()}")

In [ ]:
import sys, copy
import numpy as np
import torch
import torch.nn as nn
import yaml
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "vjepa2"))

import app.vjepa_2_1.models.predictor as vit_pred
import app.vjepa_2_1.models.vision_transformer as video_vit
from app.vjepa_2_1.wrappers import MultiSeqWrapper, PredictorMultiSeqWrapper
from data_pipeline.patch_embed_6ch import build_nch_patch_embed_from_pretrained
from data_pipeline.olmoearth_dataset import OLMoEarthDataset
from src.masks.multiseq_multiblock3d import MaskCollator
from src.masks.utils import apply_masks
from src.utils.schedulers import CosineWDSchedule, WarmupCosineSchedule

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16
print(f"Device: {device}")

## 2. Build model + load checkpoint

In [ ]:
with open(CONFIG_YML) as f:
    cfg = yaml.safe_load(f)
d, m, oe = cfg["data"], cfg["model"], cfg["olmoearth"]

def _strip(sd):
    if not sd: return sd
    k = next(iter(sd))
    for pfx in ("module.backbone.", "module.", "backbone."):
        if k.startswith(pfx):
            return {k[len(pfx):]: v for k, v in sd.items()}
    return sd

def _load_compat(module, sd):
    own  = module.state_dict()
    ok   = {k: v for k, v in sd.items() if k not in own or own[k].shape == v.shape}
    return module.load_state_dict(ok, strict=False)

enc_bb = video_vit.__dict__[m["model_name"]](
    img_size=d["crop_size"], patch_size=d["patch_size"],
    num_frames=d["frames_per_clip"], tubelet_size=d["tubelet_size"],
    in_chans=m["in_chans"], use_doy_encoding=m.get("use_doy_encoding", True),
    use_rope=m.get("use_rope", True), uniform_power=True, use_sdpa=True,
    use_activation_checkpointing=False, modality_embedding=False,
    has_cls_first=False, n_registers=0,
)
pred_bb = vit_pred.vit_predictor(
    img_size=d["crop_size"], patch_size=d["patch_size"],
    num_frames=d["frames_per_clip"], tubelet_size=d["tubelet_size"],
    embed_dim=enc_bb.embed_dim, predictor_embed_dim=m.get("pred_embed_dim", 384),
    depth=m.get("pred_depth", 12), num_heads=m.get("pred_num_heads", 12),
    uniform_power=True, use_mask_tokens=True, zero_init_mask_tokens=True,
    use_rope=m.get("use_rope", True), use_sdpa=True,
    use_activation_checkpointing=False, n_registers=0,
    has_cls_first=False, modality_embedding=False,
)
encoder        = MultiSeqWrapper(enc_bb).to(device)
predictor      = PredictorMultiSeqWrapper(pred_bb).to(device)
target_encoder = copy.deepcopy(encoder)
for p in target_encoder.parameters(): p.requires_grad = False

# Load pretrained weights — Prithvi-style N-ch patch_embed init
ckpt      = torch.load(CKPT_PATH, map_location="cpu")
enc_state = _strip(ckpt.get("encoder", ckpt))

new_pe = build_nch_patch_embed_from_pretrained(
    enc_state, in_chans=m["in_chans"], patch_size=d["patch_size"],
    tubelet_size=d["tubelet_size"], embed_dim=enc_bb.embed_dim,
).to(device)
enc_bb.patch_embed = new_pe

enc_filtered = {k: v for k, v in enc_state.items() if not k.startswith("patch_embed")}
msg = _load_compat(enc_bb, enc_filtered)
miss_imp = [k for k in msg.missing_keys
            if not k.startswith(("doy_encoding", "patch_embed", "pos_embed"))]
assert not miss_imp, f"Important keys missing: {miss_imp}"

if "predictor" in ckpt:
    _load_compat(pred_bb, _strip(ckpt["predictor"]))

print(f"✓ {sum(p.numel() for p in encoder.parameters()):,} encoder params")
print(f"✓ Checkpoint: {Path(CKPT_PATH).name}")

## 3. Dataset + DataLoader

In [ ]:
dataset = OLMoEarthDataset(
    tar_path=TAR_GLOB,
    n_bands_per_timestep=oe.get("n_bands_per_timestep", 4),
    crop_size=d["crop_size"],
    dn_scale=oe.get("dn_scale", 10000.0),
    max_missing_frac=oe.get("max_missing_frac", 0.10),
    shuffle_buffer=oe.get("shuffle_buffer", 1000),
)
mask_collator = MaskCollator(
    cfgs_mask=cfg["mask"],
    dataset_fpcs=[d["frames_per_clip"]],
    crop_size=(d["crop_size"], d["crop_size"]),
    patch_size=(d["patch_size"], d["patch_size"]),
    tubelet_size=d["tubelet_size"],
)
nw = d.get("num_workers", 4)
loader = torch.utils.data.DataLoader(
    dataset, collate_fn=mask_collator, batch_size=d["batch_size"],
    shuffle=False, drop_last=True, num_workers=nw,
    pin_memory=d.get("pin_mem", True), persistent_workers=(nw > 0),
)
try:
    ipe = len(loader)
except TypeError:
    ipe = d.get("ipe", 17830)
print(f"Dataset: {len(dataset.tar_files)} TAR shard(s)   ipe={ipe}   batch_size={d['batch_size']}")
print(f"Collator: {len(cfg['mask'])} generators (Gen0: 8 small 15% blocks, Gen1: 2 large 70% blocks)")

## 4. Sanity check

Run one forward pass **without** gradient to confirm:
- loss ≈ 0.29 (pretrained encoder)
- loss ≈ 0.47 (random init — no checkpoint loaded)

This verifies the full pipeline before launching training.

In [ ]:
encoder.eval(); predictor.eval(); target_encoder.eval()
_it = iter(loader)
_fpc = next(_it)

for _cb, _me_raw, _mp_raw in _fpc:
    _x    = _cb[0][0].to(device, dtype=dtype)
    _doys = _cb[2].to(device)
    _me   = [[m.to(device) for m in _me_raw]]
    _mp   = [[m.to(device) for m in _mp_raw]]

    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=dtype):
        _z_ctx     = encoder([_x], masks=_me, doys=_doys, training_mode=True)
        _z_tgt_full = target_encoder([_x], doys=_doys, training_mode=True)
        _z_tgt     = [[apply_masks(z, [m]) for m in mp]
                      for z, mp in zip(_z_tgt_full, _mp)]
        _z_pred, _ = predictor(_z_ctx, _me, _mp)
        _n  = sum(len(l) for l in _z_pred)
        _loss = sum(nn.functional.smooth_l1_loss(zp, zt)
                    for zpl, ztl in zip(_z_pred, _z_tgt)
                    for zp, zt in zip(zpl, ztl)) / max(_n, 1)

print(f"Sanity loss: {_loss.item():.4f}  (pretrained baseline ≈ 0.29, random init ≈ 0.47)")
assert not torch.isnan(_loss), "NaN loss — check checkpoint + data paths"
print("✓ Sanity check passed")
del _it, _fpc, _cb, _me_raw, _mp_raw, _x, _doys, _me, _mp
del _z_ctx, _z_tgt_full, _z_tgt, _z_pred, _loss

## 5. Training

3-stage schedule from `olmoearth-256px-12f.yaml`:

| Stage | Epochs | LR | Frozen |
|-------|--------|----|--------|
| 1 | 20 | 1e-3 | backbone; patch_embed + doy_encoding trainable |
| 2 | 30 | 5e-5 | backbone; last 6 blocks unfrozen |
| 3 | 50 | 1e-5 | full fine-tune |

Set `STEPS_CAP=50` (Cell 1) for a quick smoke test, `None` for full training.

In [ ]:
def _apply_freeze(enc, stage_cfg):
    freeze = stage_cfg.get("freeze_backbone", False)
    n_uf   = stage_cfg.get("unfreeze_last_n_blocks", -1)
    bb = enc.backbone
    for p in bb.parameters(): p.requires_grad = not freeze
    for p in bb.patch_embed.parameters(): p.requires_grad = True
    if bb.doy_encoding is not None:
        for p in bb.doy_encoding.parameters(): p.requires_grad = True
    if freeze and n_uf > 0:
        for blk in list(bb.blocks)[-n_uf:]:
            for p in blk.parameters(): p.requires_grad = True
    tr = sum(p.numel() for p in enc.parameters() if p.requires_grad)
    tt = sum(p.numel() for p in enc.parameters())
    return tr, tt

def _make_opt(enc, pred, scfg, ipe):
    lr = scfg["lr"]
    def _split(mod):
        wd  = [p for n, p in mod.named_parameters() if p.requires_grad and "bias" not in n and p.ndim != 1]
        nwd = [p for n, p in mod.named_parameters() if p.requires_grad and ("bias" in n or p.ndim == 1)]
        return wd, nwd
    ew, en = _split(enc); pw, pn = _split(pred)
    opt = torch.optim.AdamW(
        [{"params": ew + pw}, {"params": en + pn, "weight_decay": 0}],
        lr=lr, weight_decay=scfg.get("weight_decay", 0.05),
    )
    T   = scfg["epochs"] * ipe
    sch = WarmupCosineSchedule(
        opt, warmup_steps=scfg.get("warmup", 0) * ipe,
        start_lr=scfg.get("start_lr", lr), ref_lr=lr,
        final_lr=scfg.get("final_lr", 1e-6), T_max=T,
    )
    wds = CosineWDSchedule(
        opt, ref_wd=scfg.get("weight_decay", 0.05),
        final_wd=scfg.get("final_weight_decay", 0.05), T_max=T,
    )
    return opt, sch, wds

In [ ]:
scaler       = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
ema          = cfg["optimization"]["ema"][0]
loss_exp     = cfg["loss"].get("loss_exp", 1.0)
loss_history = []      # [(global_step, loss_value), ...]
global_epoch = 0

for stage_name, stage_cfg in [
    ("stage1", cfg["optimization"]["stage1"]),
    ("stage2", cfg["optimization"]["stage2"]),
    ("stage3", cfg["optimization"]["stage3"]),
]:
    tr, tt = _apply_freeze(encoder, stage_cfg)
    print(f"\n{'='*58}\n  {stage_name.upper()}  epochs={stage_cfg['epochs']}  "
          f"trainable={tr:,}/{tt:,} ({100*tr/tt:.1f}%)\n{'='*58}")

    opt, sch, wds = _make_opt(encoder, predictor, stage_cfg, ipe)

    for ep in range(stage_cfg["epochs"]):
        encoder.train(); predictor.train(); target_encoder.eval()
        pbar = tqdm(loader, total=STEPS_CAP or ipe,
                    desc=f"{stage_name} ep{global_epoch:03d}")
        ep_losses = []

        for step, fpc_collations in enumerate(pbar):
            if STEPS_CAP and step >= STEPS_CAP:
                break

            for cb, me_raw, mp_raw in fpc_collations:
                x    = cb[0][0].to(device, dtype=dtype, non_blocking=True)
                doys = cb[2].to(device, non_blocking=True)
                # MaskCollator returns [gen0[B,K0], gen1[B,K1]];
                # wrap in clip list: [[gen0, gen1]] (outer=clips, inner=generators)
                me   = [[m.to(device) for m in me_raw]]
                mp   = [[m.to(device) for m in mp_raw]]

                opt.zero_grad()
                with torch.autocast(device_type=device.type, dtype=dtype):
                    z_ctx      = encoder([x], masks=me, doys=doys, training_mode=True)
                    with torch.no_grad():
                        z_tgt_full = target_encoder([x], doys=doys, training_mode=True)
                        z_tgt = [[apply_masks(z, [m]) for m in mpl]
                                 for z, mpl in zip(z_tgt_full, mp)]
                    z_pred, _ = predictor(z_ctx, me, mp)
                    n_terms = sum(len(l) for l in z_pred)
                    loss = sum(
                        nn.functional.smooth_l1_loss(zp, zt) ** loss_exp
                        for zpl, ztl in zip(z_pred, z_tgt)
                        for zp, zt in zip(zpl, ztl)
                    ) / max(n_terms, 1)

                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(
                    list(encoder.parameters()) + list(predictor.parameters()), 1.0)
                scaler.step(opt); scaler.update()
                sch.step(); wds.step()

                with torch.no_grad():
                    for pe, pt in zip(encoder.parameters(), target_encoder.parameters()):
                        pt.data.mul_(ema).add_((1 - ema) * pe.data)

                v = loss.item()
                ep_losses.append(v)
                loss_history.append(v)
                pbar.set_postfix(loss=f"{v:.4f}")

        avg = sum(ep_losses) / max(len(ep_losses), 1)
        print(f"  ep {global_epoch:03d}  avg_loss={avg:.4f}")

        if global_epoch % 10 == 0:
            torch.save({
                "epoch": global_epoch, "stage": stage_name,
                "encoder": encoder.backbone.state_dict(),
                "predictor": predictor.backbone.state_dict(),
            }, f"{OUTPUT_DIR}/ckpt_ep{global_epoch:04d}.pth")
        global_epoch += 1

print("\n✓ Training complete")
torch.save({
    "epoch": global_epoch,
    "encoder":   encoder.backbone.state_dict(),
    "predictor": predictor.backbone.state_dict(),
}, f"{OUTPUT_DIR}/ckpt_final.pth")
print(f"Final checkpoint → {OUTPUT_DIR}/ckpt_final.pth")

## 6. Loss curve

In [ ]:
if loss_history:
    arr = np.array(loss_history)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(arr, lw=0.6, alpha=0.5, color="steelblue", label="step loss")
    w = min(100, len(arr) // 5 + 1)
    if len(arr) >= w:
        smooth = np.convolve(arr, np.ones(w) / w, mode="valid")
        ax.plot(range(w - 1, len(arr)), smooth, lw=2, color="tomato",
                label=f"MA({w})")
    ax.set_xlabel("step (all stages combined)")
    ax.set_ylabel("smooth-L1 loss")
    ax.set_title("Fine-Tuning Loss History")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Last 100 steps avg: {arr[-100:].mean():.4f}")
else:
    print("No training history yet — run the training cell first.")

## 7. CLI command for production training

For full training (100 epochs, ~272 h on 1×A100), set the paths in the YAML and run:

In [ ]:
print("# 1. Edit yaml paths:")
print(f"#    tar_path: {TAR_GLOB}")
print(f"#    pretrained_checkpoint: {CKPT_PATH}")
print()
print(f"python finetune_main.py --config {CONFIG_YML}")
print()
print("# Background with logging:")
print(f"nohup python finetune_main.py --config {CONFIG_YML} > train.log 2>&1 &")

## 8. Post-training: load fine-tuned checkpoint + PCA embeddings

Load the fine-tuned weights and extract full-sequence embeddings from one batch.
PCA(3) projected to RGB shows how the fine-tuned encoder organises seasonal land-cover features.

In [ ]:
FINETUNE_CKPT = f"{OUTPUT_DIR}/ckpt_final.pth"

if Path(FINETUNE_CKPT).exists():
    ft_ckpt = torch.load(FINETUNE_CKPT, map_location="cpu")
    msg_ft  = _load_compat(enc_bb, ft_ckpt.get("encoder", ft_ckpt))
    print(f"Loaded: ep={ft_ckpt.get('epoch', '?')}  "
          f"missing={len(msg_ft.missing_keys)}  unexpected={len(msg_ft.unexpected_keys)}")
else:
    print(f"Checkpoint not found: {FINETUNE_CKPT}")
    print("Run the training cell (Section 5) first, or point FINETUNE_CKPT at an existing file.")

In [ ]:
# Full-sequence embeddings — no masking, eval mode
encoder.eval()
_it2 = iter(loader)
_fpc2 = next(_it2)

for _cb, _me_raw, _mp_raw in _fpc2:
    _x    = _cb[0][0].to(device, dtype=dtype)
    _doys = _cb[2].to(device)
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=dtype):
        # training_mode=False → [B, N_TOK, D] (single clip output)
        z_full = encoder([_x], doys=_doys, training_mode=False)[0]
    break
del _it2, _fpc2, _cb, _me_raw, _mp_raw, _x, _doys

# Use first 1024 dims (last-block features) for PCA
z_np = z_full[0, :, :1024].float().cpu().numpy()    # [N_TOK, 1024]
print(f"Embedding shape: {z_np.shape}")

# PCA(3) via NumPy eigh
z_c = z_np - z_np.mean(axis=0)
cov = z_c.T @ z_c / (z_np.shape[0] - 1)
eigvals, eigvecs = np.linalg.eigh(cov)        # ascending order
top3   = eigvecs[:, -3:][:, ::-1]             # descending variance
pca3   = z_c @ top3                           # [N_TOK, 3]
for i in range(3):
    lo, hi = np.percentile(pca3[:, i], 1), np.percentile(pca3[:, i], 99)
    pca3[:, i] = np.clip((pca3[:, i] - lo) / (hi - lo + 1e-8), 0, 1)

var_exp = eigvals[-3:] / eigvals.sum()

# Reshape to (T_PAT, H_PAT, W_PAT, 3) and display
T_PAT = d["frames_per_clip"] // d["tubelet_size"]     # 6
H_PAT = W_PAT = d["crop_size"] // d["patch_size"]     # 16
pca_grid = pca3.reshape(T_PAT, H_PAT, W_PAT, 3)

TUBELET_LABELS = ["Jan-Feb", "Mar-Apr", "May-Jun", "Jul-Aug", "Sep-Oct", "Nov-Dec"]
fig, axes = plt.subplots(1, T_PAT, figsize=(T_PAT * 2.6, 3))
for ti in range(T_PAT):
    up = np.repeat(np.repeat(pca_grid[ti], 16, axis=0), 16, axis=1)
    axes[ti].imshow(up)
    axes[ti].set_title(TUBELET_LABELS[ti], fontsize=9)
    axes[ti].axis("off")
fig.suptitle(
    f"Fine-tuned Encoder — Patch Embeddings PCA(3)\n"
    f"PC1={var_exp[2]*100:.1f}%  PC2={var_exp[1]*100:.1f}%  PC3={var_exp[0]*100:.1f}% var",
    fontsize=11,
)
plt.tight_layout()
plt.show()